# Работа с данными кодов IATA

Коды ИАТА (IATA — International Air Transport Association) — это индивидуальные идентификаторы объектов, имеющих значение для индустрии пассажирских авиаперевозок. Они присваиваются Международной ассоциацией воздушного транспорта.

Эти коды нужны для обращения к API "Яндекс Расписаний"

Для данной работы был использован датасет с Kaggle, который содержит более 9000 подобных наименований: https://www.kaggle.com/datasets/zinovadr/iata-airport-code

## Импорт и перевод кодов

In [1]:
import pandas as pd
import numpy as np
from tqdm import tqdm
from functools import lru_cache

tqdm.pandas()

In [2]:
df_iata = pd.read_csv('../data/airport-codes_csv.csv')
df_iata = df_iata[df_iata['iata_code'].notna()][['iata_code', 'name', 'iso_country', 'municipality']].rename(columns={'name': 'airport_name_en', 'municipality': 'municipality_en'}).reset_index(drop=True)
df_iata.head()

,iata_code,airport_name_en,iso_country,municipality_en
0,UTK,Utirik Airport,MH,Utirik Island
1,OCA,Ocean Reef Club Airport,US,Key Largo
2,PQS,Pilot Station Airport,US,Pilot Station
3,CSE,Crested Butte Airpark,US,Crested Butte
4,JCY,LBJ Ranch Airport,US,Johnson City


In [3]:
from deep_translator import GoogleTranslator

translator = GoogleTranslator(source='en', target='ru')

@lru_cache(maxsize=1000)
def translator_error_handle(phrase: str) -> str:
    result = None
    try:
        result = translator.translate(phrase)
        return result
    except Exception:
        return np.nan
    

In [4]:
print('Старт перевода названий аэропортов...')
df_iata['airport_name_ru'] = df_iata['airport_name_en'].progress_apply(translator_error_handle)
print('Успешно!')

Старт перевода названий аэропортов...


100%|██████████| 9199/9199 [3:47:36<00:00,  1.48s/it]  

Успешно!


In [6]:
df_iata.to_csv('../data/iata_en_rus_autotranslate.csv', index=False)

In [7]:
print('Старт перевода местоположений аэропортов...')
df_iata['municipality_ru'] = df_iata['municipality_en'].progress_apply(translator_error_handle)
print('Успешно!')

Старт перевода местоположений аэропортов...


100%|██████████| 9199/9199 [3:43:31<00:00,  1.46s/it]  

Успешно!


In [8]:
df_iata.to_csv('../data/iata_en_rus_autotranslate.csv', index=False)

In [9]:
df_iata.head()

,iata_code,airport_name_en,iso_country,municipality_en,airport_name_ru,municipality_ru
0,UTK,Utirik Airport,MH,Utirik Island,Утирик аэропорт,Остров Утирик
1,OCA,Ocean Reef Club Airport,US,Key Largo,Оушен Риф Клуб Аэропорт,Ки Ларго
2,PQS,Pilot Station Airport,US,Pilot Station,Пилотная станция Аэропорт,Пилотная станция
3,CSE,Crested Butte Airpark,US,Crested Butte,Аэропорт Крестед-Бьютт,Крестед Бьютт
4,JCY,LBJ Ranch Airport,US,Johnson City,LBJ Ранчо аэропорт,Джонсон Сити


## Приведение значений и работа с непереведенными данными

In [ ]:
df_iata = pd.read_csv('../data/iata_en_rus_autotranslate.csv')

In [10]:
df_iata.isna().sum()

iata_code            0
airport_name_en      0
iso_country         31
municipality_en    760
airport_name_ru      1
municipality_ru    765
dtype: int64

In [13]:
df_iata[df_iata['municipality_ru'].isna()]

,iata_code,airport_name_en,iso_country,municipality_en,airport_name_ru,municipality_ru
74,AEE,Adareil Airport,SS,NaN,Адареил аэропорт,NaN
81,AFR,Afore Airstrip,PG,NaN,Перед взлетно-посадочной полосой,NaN
90,CHY,Choiseul Bay Airport,SB,NaN,Аэропорт Шуазель Бэй,NaN
95,AVU,Avu Avu Airport,SB,NaN,Аву Аву аэропорт,NaN
98,MUA,Munda Airport,SB,NaN,Аэропорт Мунда,NaN
...,...,...,...,...,...,...
9058,UUN,Baruun Urt Airport,MN,NaN,Аэропорт Баруун Урт,NaN
9059,COQ,Choibalsan Airport,MN,NaN,Аэропорт Чойбалсан,NaN
9063,KHR,Kharkhorin Airport,MN,NaN,Аэропорт Хархорин,NaN
9071,ULG,Ulgii Mongolei Airport,MN,NaN,Аэропорт Улгий Монголей,NaN


## Добавление таблицы в БД

In [15]:
import sqlite3

conn = sqlite3.connect('../data/flights_info.db')
df_iata.to_sql('airports_info', conn, if_exists='replace', index=False)

df_loaded = pd.read_sql('SELECT * FROM airports_info', conn)
conn.close()

In [16]:
df_loaded

,iata_code,airport_name_en,iso_country,municipality_en,airport_name_ru,municipality_ru
0,UTK,Utirik Airport,MH,Utirik Island,Утирик аэропорт,Остров Утирик
1,OCA,Ocean Reef Club Airport,US,Key Largo,Оушен Риф Клуб Аэропорт,Ки Ларго
2,PQS,Pilot Station Airport,US,Pilot Station,Пилотная станция Аэропорт,Пилотная станция
3,CSE,Crested Butte Airpark,US,Crested Butte,Аэропорт Крестед-Бьютт,Крестед Бьютт
4,JCY,LBJ Ranch Airport,US,Johnson City,LBJ Ранчо аэропорт,Джонсон Сити
...,...,...,...,...,...,...
9194,DLC,Zhoushuizi Airport,CN,Dalian,Аэропорт Чжоушуйцзы,Далянь
9195,TNH,Tonghua Sanyuanpu Airport,CN,Tonghua,Аэропорт Тунхуа Саньюаньпу,Тунхуа
9196,SHE,Taoxian Airport,CN,Shenyang,Таосянь аэропорт,Шэньян
9197,YNJ,Yanji Chaoyangchuan Airport,CN,Yanji,Аэропорт Яньцзи Чаоянчуань,Яньцзи
